# Sumarização extrativa de texto baseada em frequência de termos

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK  
- **Arquivo Original:** `sumarizacao_freq.ipynb`  

---


## Sumarização Automática de Textos
## Exemplo de sumarização baseada em frequência.
*   Adaptação por Emerson Cabrera Paraiso e Lucas Oliveira
*   Algoritmo original por Akash Panchal e disponível em https://becominghuman.ai/text-summarization-in-5-steps-using-nltk-65b21e352b65

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('rslp')

### Função que cria a tabela de frequência das palavras
Todas palavras passam pelo algoritmo de Stemming. Stopwords são retiradas, e em seguida as frequências das palavras são calculadas.

In [ ]:
def tabela_de_frequencia(text_string) -> dict:
    stopWords = set(stopwords.words("portuguese"))
    words = word_tokenize(text_string)
    stemmer = nltk.stem.RSLPStemmer()

    freqTable = dict()
    for word in words:
        word = stemmer.stem(word)
        if word in stopWords:
            continue
        # Se palavra já está na tabela
        if word in freqTable:
            # Incrementa sua frequencia
            freqTable[word] += 1
        else:
            # Cria posição da palavra na tabela
            freqTable[word] = 1

    return freqTable

### Função que define a pontuação das sentenças
Para cada sentença calcula-se a pontuação baseado nas palavras da sentença que estão presentes na tabela de frequências.
Exemplo:
```
freqTable = {'teste':5, 'sumarização':2}
sentence = "Este é um teste de sumarização"
```
Para a sentença acima a pontuação será 7, pois a palavra 'teste' tem valor 5 e a palavra 'sumarização' 2.


Ao final, a pontuação da sentença é divida pela quantidade de palavras da sentença, para evitar que sentenças longas levem vantagem sobre sentenças curtas.



In [ ]:
def pontuar_sentencas(sentences, freqTable) -> dict:
    sentenceValue = dict()

    # Percorre as sentenças
    for sentence in sentences:
        word_count_in_sentence = (len(word_tokenize(sentence)))
        word_count_in_sentence_except_stop_words = 0

        # Percorre as palavras na tabela de frequência
        for wordValue in freqTable:
            # Caso encontre a palavra na sentença atual
            if wordValue in sentence.lower():
                # Incrementa contador
                word_count_in_sentence_except_stop_words += 1
                # Se a sentença atual já tem pontuação atribuida
                if sentence[:10] in sentenceValue:
                    # Soma nova pontuação
                    sentenceValue[sentence[:10]] += freqTable[wordValue]
                else:
                    # Atribui valor da palavra atual para a sentença
                    sentenceValue[sentence[:10]] = freqTable[wordValue]

        # Se a sentença atual teve pontuação atribuida
        if sentence[:10] in sentenceValue:
            # Dividimos sua pontuação pela quantidade de palavras da sentença - para evitar que sentenças longas levem vantagem sobre sentenças curtas.
            sentenceValue[sentence[:10]] = sentenceValue[sentence[:10]] / word_count_in_sentence_except_stop_words

    return sentenceValue

### Função que calcula a pontuação média de todas as sentenças
Calcula a pontuação média de todas as sentenças

In [ ]:
def calcula_pontuacao_media(pontuacao_sentencas) -> int:

    sumValues = 0
    for entry in pontuacao_sentencas:
        sumValues += pontuacao_sentencas[entry]

    average = (sumValues / len(pontuacao_sentencas))

    return average

### Função que gera o sumário baseado nas sentenças com pontuação maior que o threshold definido

In [ ]:
def gerar_sumario(sentencasOriginais, pontuacao_sentencas, threshold):
    contador_sentencas = 0
    sumario = ''

    for sentencaOriginal in sentencasOriginais:
        if sentencaOriginal[:10] in pontuacao_sentencas and pontuacao_sentencas[sentencaOriginal[:10]] >= (threshold):
            sumario += " " + sentencaOriginal
            contador_sentencas += 1

    return sumario

### Exemplos de textos de entrada

In [ ]:
# Texto 1
text = """Dezenas de veículos foram incendiados em frente a sede do governo da região de Guerrero, no México, em um protesto pelo desaparecimento e morte de 43 estudantes da escola normal rural de Ayotzinapa.

Mais de 300 jovens, a maioria com o rosto coberto, atacaram a fachada do edifício em Chipancingo, capital de Guerrero.

O protesto ocorreu após o procurador-geral da República do país, Jesús Murillo Karam, informar que três homens suspeitos de ser integrantes do cartel Guerreros Unidos confessaram ter matado os estudantes e queimado seus corpos. O presidente Enrique Peña Nieto prometeu na sexta (7) punir todos os responsáveis pelos “crimes abomináveis”.

Os jovens sumiram em 26 de setembro, depois de arrecadar fundos para a escola em Iguala (a 192 km da Cidade do México).

Na saída da cidade, dois ônibus que voltavam à instituição com os alunos foram alvejados por policiais e traficantes do Guerreros Unidos. O ataque deixou seis mortos.

Os confessores, identificados como Particio Reyes, Jonathan Osorio e Agustín García Reyes, dizem que receberam os 43 estudantes no lixão de Cocula, a 22 km de Iguala. Segundo os pistoleiros, 15 deles chegaram ao local mortos com sinais de asfixia.

Segundo a Procuradoria-Geral do México, os detidos não disseram quem levou os estudantes e quem era o mandante da emboscada.

O órgão, porém, acredita que o mandante foi o prefeito de Iguala, Jose Luis Abarca, preso na quarta (5).

A intenção seria evitar que os alunos atrapalhassem um evento em que sua mulher, María de los Ángeles Pineda, seria lançada como candidata a sucedê-lo. A mulher de Abarca é irmã de três chefes do Guerreros Unidos.

FAMILIARES

Em entrevista, os parentes disseram não acreditar na versão do procurador-geral e pediram que o material recolhido seja analisado por peritos independentes.

Para eles, o governo quer fazer com que eles acreditem que seus filhos estão mortos. “Sequer mostraram fotos dos nossos filhos. Enquanto não houver provas, nossos filhos estão vivos”, disse Felipe de la Cruz, pai de um dos alunos.

Os pais pediram ao governo que prossiga com as buscas e permita a assistência técnica da Comissão Interamericana de Direitos Humanos."""



In [ ]:
# Texto 2
text = '''O ministro Alexandre de Moraes, do Supremo Tribunal Federal (STF), validou nesta terça-feira (17) o acordo para utilização do fundo bilionário da Petrobras – de cerca de R$ 2,6 bilhões.

O acordo foi assinado pelo governo federal, representantes de Câmara e Senado e da Procuradoria Geral da República (PGR) e enviado para homologação pelo ministro. Moraes validou a seguinte divisão para uso do fundo:

1) R$ 1,601 bilhão para educação, sendo:

R$ 1 bilhão para ações relacionadas à educação infantil;
R$ 250 milhões para o Ministério da Cidadania, para ações relacionadas ao Programa Criança Feliz (desenvolvimento integral da Primeira Infância);
R$ 250 milhões para o Ministério da Ciência, Tecnologia, Inovações e Comunicações, para ações de empreendedorismo e bolsas do Conselho Nacional de Desenvolvimento Científico e Tecnológico (CNPq);
R$ 100 milhões para ações socioeducativas em cooperação com os Estados, preferencialmente por intermédio do Ministério da Mulher, da Família e dos Direitos Humanos.
2) R$ 1,060 bilhão para Amazônia (prevenção, fiscalização e combate a desmatamentos e incêndios), sendo:

R$ 630 milhões para administração pela União de ações de operações de Garantia da Lei e da Ordem (GLO) e outros institutos;
R$ 430 milhões para execução pelos estados da região amazônica.
Na decisão, o ministro determinou a imediata transferência dos recursos financeiros depositados, devidamente corrigidos, para uma conta única do Tesouro Nacional para que o acordo seja cumprido integralmente.

Com relação ao uso R$ 430 milhões na Amazônia legal, Moraes afirmou que a utilização dos recursos deve observar:

área territorial do Estado;
população estimada na data da homologação do acordo;
o inverso do PIB per capita dos Estados, o número de focos de queimadas;
a área desmatada total por Estado.
O ministro também concedeu pedido da Caixa para que a instituição desconte a remuneração relativa ao tempo em que permaneceu com a guarda dos valores.


Alexandre de Moraes declarou ainda a nulidade do acordo firmado entre a força-tarefa da Lava Jato e a Petrobras. O ministro já havia suspenso a eficácia do acordo em março.

"A eventual apropriação, por determinados membros do Ministério Público, da administração e destinação de proveito econômico resultante da atuação do órgão, além de desrespeitar os princípios da legalidade, da impessoalidade e da moralidade administrativa, implicou séria agressão ao perfil constitucional fortalecido da Instituição", afirmou o ministro.

Salles e governadores definem critérios para uso do Fundo Petrobras

Regras para uso do dinheiro
O ministro do Supremo validou as regras para uso dos recursos do fundo definidas no acordo. São elas:

o saldo corrigido será transferido para a conta única do Tesouro Nacional;
os recursos devem ser convertidos em receita da União, que se comprometerá a fazer com que passem a compor fonte de recursos específica, passível de acompanhamento pelos órgãos de controle;
foi publicada a portaria “Recursos Oriundos de Leis ou Acordos Anticorrupção” para assegurar transparência e controle quando os valores forem utilizados para financiar as despesas públicas do acordo;
a União, por meio do Ministério da Economia, se compromete a garantir as condições necessárias para execução total das despesas neste exercício financeiro e nos seguintes, até o completo exaurimento dos valores;
a realização das despesas será concretizada especialmente por meio da concessão de limite de empenho e de movimentação financeira (“descontingenciamento”) e da abertura de créditos adicionais;
os valores destinados aos Estados da região amazônica serão divididos mediante critérios a serem definidos pelos ministérios.

Acordo da Petrobras
Para encerrar investigações sobre a empresa nos Estados Unidos, a Petrobras acordou com autoridades norte-americanas o pagamento de US$ 853,2 milhões. Desse valor, US$ 682 milhões devem ser aplicados no Brasil - cerca de R$ 2,6 bilhões.

O valor é fruto de reparação de danos admitidos pela Petrobras, e não da devolução de valores com delações premiadas.

O montante foi depositado em uma conta judicial, mas está bloqueado.

Inicialmente, a força-tarefa da Lava Jato havia feito um acordo com a Petrobras para criar um fundo privado com parte dinheiro. Mas o ministro do STF, Alexandre de Moraes, suspendeu o ato.'''


### Pipeline de execução principal

###1) Cria tabela de frequência

In [ ]:
tabela_freq = tabela_de_frequencia(text)

###2) Segmenta as sentenças

In [ ]:
sentencas = sent_tokenize(text)

### 3) Pontuação das sentenças

In [ ]:
pontuacao_sentencas = pontuar_sentencas(sentencas, tabela_freq)

###4) Definição de threshold
Média da pontuação das sentenças

In [ ]:
threshold = calcula_pontuacao_media(pontuacao_sentencas)

###5) Gera o sumário

In [ ]:
sumario = gerar_sumario(sentencas, pontuacao_sentencas, threshold)
print(sumario)

## Atividades

1. Execute a sumarização do texto 1 e analise a tabela de frequência. Você considera que os tokens com maior peso realmente são os mais importantes?
2. Verifique os resultados da sumarização ao alterarmos o valor do threshold (e.g., aumentar e diminuir em 50%, 75%, 100%). Após análise responda: o que você acha do uso de um threshold baseado em uma pontuação? Funcionaria bem para todos casos de sumarização?
3. Faça uma implementação do algoritmo em que o tamanho do sumário final seja definido por um threshold que estabeleça um número fixo de sentenças. Quais benefícios e malefícios desta opção?
4. Faça uma implementação do algoritmo em que o tamanho do sumário final seja definido por um threshold que estabeleça que o sumário final deve ter no máximo 10% do original. Quais benefícios e malefícios desta opção?

## Desafios
*   Implementar função de filtragem das sentenças, para retirar sentenças com alto grau de similaridade semântica (pode-se fazer uso de WordNets, Word Embeddings, etc.). Antes de inserir uma nova sentença na lista de selecionadas, verificar similaridade da mesma com as previamente selecionadas. Caso tenha alto grau de similaridade (defina um threshold), descartar.
*   Faça uma implementação que ao invés de usar apenas a frequência simples como fator de importância, que utilize o TF-IDF